# Validasi Digital Checkup — Fuzzy AHP + TOPSIS

Notebook ini menghitung ulang **secara independen** (Python standar, tanpa memanggil kode web) setiap hasil Digital Checkup yang diekspor dari halaman admin, lalu membandingkannya dengan jejak hitung yang disimpan web.

**Cara pakai**

1. Di `/admin` → tab **Ekspor** → *Ekspor teknis untuk validasi metode* → **JSON siap Google Colab** (semua UMKM), atau tombol **Ekspor input U00x untuk Colab** pada kartu satu UMKM.
2. Di Colab pilih **Runtime → Run all**, lalu unggah file `digital-checkup-colab.json` saat diminta.
3. Di luar Colab: `DL_COLAB_JSON=/path/digital-checkup-colab.json jupyter nbconvert --execute ...` atau jalankan sel-selnya dengan variabel lingkungan yang sama.

**Kriteria lulus (per record berstatus `RANKED`)**

- Selisih mutlak maksimum `|web − notebook| ≤ 1e-10` untuk `w, R, S, g, H, q, N, V, A+, A−, D+, D−, C`.
- `status`, `rank`, `topSet`, dan urutan kode modul pada ranking sama persis.
- Hash SHA-256 input canonical sama dengan `inputHash` yang disimpan web.

Record yang bukan `RANKED` (tanpa gap, dimensi tidak terwakili, alternatif identik) dan draf yang belum dikirim dicantumkan di ringkasan tetapi tidak dihitung lulus/gagal. Notebook ini **tidak membuat data contoh**; seluruh angka berasal dari file ekspor.

## 1. Mesin hitung independen
Implementasi ulang Fuzzy AHP (geometric mean tiga ahli, TFN Saaty, centroid) dan TOPSIS personal (`g = 1 − S/100`).

In [ ]:
import hashlib, json, math, os

EPS = 1e-12      # sama dengan EPSILON di lib/decision-engine
TOL = 1e-10      # batas selisih mutlak per record
RI8 = 1.41       # random index Saaty untuk n = 8
D = ['C%d' % i for i in range(1, 9)]
M = ['A%d' % i for i in range(1, 8)]
NUMERIC_KEYS = ['w', 'R', 'S', 'g', 'H', 'q', 'N', 'V', 'idealPlus', 'idealMinus', 'Dplus', 'Dminus', 'C']

def geo(values):
    return math.exp(sum(math.log(x) for x in values) / len(values))

def normalize(values):
    total = sum(values)
    assert total > EPS, 'Jumlah bobot harus positif.'
    return [x / total for x in values]

def fuzzy(x):
    # TFN Saaty: k -> (max(1, k-1), k, min(9, k+1)); resiprokal membalik urutan komponen.
    if x == 1:
        return [1., 1., 1.]
    for k in range(2, 10):
        t = [max(1, k - 1), k, min(9, k + 1)]
        if x == k:
            return t
        if x == 1 / k:
            return [1 / t[2], 1 / t[1], 1 / t[0]]
    raise ValueError('Skala pairwise harus 1, 2-9, atau resiprokalnya.')

def crisp_matrix(pairs):
    assert len(pairs) == 28, 'Setiap ahli wajib memiliki 28 pasangan.'
    matrix = [[1. for _ in D] for _ in D]
    seen = set()
    for p in pairs:
        i, j = D.index(p['left']), D.index(p['right'])
        assert i < j and (i, j) not in seen, 'Pasangan dimensi salah atau berulang.'
        seen.add((i, j))
        k = p['intensity']
        if p['preferred'] == 'equal':
            assert k == 1, 'Pilihan setara harus bernilai 1.'
            value = 1
        else:
            assert p['preferred'] in (p['left'], p['right']) and 2 <= k <= 9
            value = k if p['preferred'] == p['left'] else 1 / k
        matrix[i][j], matrix[j][i] = value, 1 / value
    return matrix

def consistency(matrix):
    # Power iteration seperti web: vektor awal 1/8, berhenti saat perubahan <= 1e-14.
    vector = [1 / 8] * 8
    for _ in range(10000):
        nxt = normalize([sum(row[j] * vector[j] for j in range(8)) for row in matrix])
        delta = max(abs(nxt[i] - vector[i]) for i in range(8))
        vector = nxt
        if delta <= 1e-14:
            break
    else:
        raise RuntimeError('Iterasi eigen tidak konvergen.')
    lam = sum(sum(row[j] * vector[j] for j in range(8)) / vector[i] for i, row in enumerate(matrix)) / 8
    ci = max(0., (lam - 8) / 7)
    return {'lambdaMax': lam, 'CI': ci, 'CR': ci / RI8, 'consistent': ci / RI8 <= .1 + EPS}

def build_model(experts):
    assert len(experts) == 3, 'Model wajib memakai tepat tiga ahli.'
    crisp = [crisp_matrix(e['pairs']) for e in experts]
    cr = [consistency(m) for m in crisp]
    assert all(c['consistent'] for c in cr), 'Ada penilaian ahli dengan CR > 0,10.'
    tfn = [[[fuzzy(x) for x in row] for row in m] for m in crisp]
    group = [[[geo([tfn[e][i][j][k] for e in range(3)]) for k in range(3)] for j in range(8)] for i in range(8)]
    row_gm = [[geo([group[i][j][k] for j in range(8)]) for k in range(3)] for i in range(8)]
    totals = [sum(row[k] for row in row_gm) for k in range(3)]
    fuzzy_w = [[row[0] / totals[2], row[1] / totals[1], row[2] / totals[0]] for row in row_gm]
    w = normalize([sum(t) / 3 for t in fuzzy_w])
    R = [[sum(e['relevance'][m][d] for e in experts) / 3 for d in D] for m in M]
    group_crisp = [[geo([crisp[e][i][j] for e in range(3)]) for j in range(8)] for i in range(8)]
    return {'w': w, 'R': R, 'fuzzyWeights': fuzzy_w, 'individualCR': cr, 'groupCR': consistency(group_crisp)}

def topsis(answers, w, R):
    assert len(answers) == 32 and all(type(v) is int and 0 <= v <= 4 for v in answers.values()), 'Jawaban harus 32 bilangan bulat 0-4.'
    S = [sum(answers['Q%02d' % (4 * j + k + 1)] for k in range(4)) / 16 * 100 for j in range(8)]
    g = [1 - s / 100 for s in S]
    H = sum(w[j] * g[j] for j in range(8))
    out = {'S': S, 'g': g, 'H': H, 'w': w, 'R': R, 'q': [0.] * 8, 'N': [], 'V': [], 'idealPlus': [], 'idealMinus': [],
           'Dplus': [], 'Dminus': [], 'C': [], 'rank': [], 'ranking': [], 'topSet': []}
    if H <= EPS:
        return dict(out, status='NO_IDENTIFIED_GAP')
    q = [w[j] * g[j] / H for j in range(8)]
    out['q'] = q
    den = [math.sqrt(sum(row[j] ** 2 for row in R)) for j in range(8)]
    if any(den[j] == 0 and g[j] > EPS for j in range(8)):
        return dict(out, status='UNCOVERED_DIMENSION')
    N = [[row[j] / den[j] if den[j] else 0. for j in range(8)] for row in R]
    V = [[row[j] * q[j] for j in range(8)] for row in N]
    hi = [max(row[j] for row in V) for j in range(8)]
    lo = [min(row[j] for row in V) for j in range(8)]
    dp = [math.sqrt(sum((row[j] - hi[j]) ** 2 for j in range(8))) for row in V]
    dm = [math.sqrt(sum((row[j] - lo[j]) ** 2 for j in range(8))) for row in V]
    out.update(N=N, V=V, idealPlus=hi, idealMinus=lo, Dplus=dp, Dminus=dm)
    if all(dp[i] + dm[i] <= EPS for i in range(7)):
        return dict(out, status='ALTERNATIVES_NOT_DISTINGUISHABLE')
    C = [dm[i] / (dp[i] + dm[i]) for i in range(7)]
    rank = [1 + sum(other > c + EPS for other in C) for c in C]   # minimum rank untuk nilai setara
    ranking = sorted(({'code': M[i], 'C': C[i], 'rank': rank[i]} for i in range(7)), key=lambda x: (x['rank'], x['code']))
    return dict(out, C=C, rank=rank, ranking=ranking, topSet=[x['code'] for x in ranking if x['rank'] == 1], status='RANKED')

def max_abs_diff(actual, expected):
    if isinstance(expected, list):
        if not isinstance(actual, list) or len(actual) != len(expected):
            return float('inf')
        return max([max_abs_diff(a, e) for a, e in zip(actual, expected)] or [0.])
    if isinstance(expected, (int, float)) and not isinstance(expected, bool):
        return abs(actual - expected)
    return 0. if actual == expected else float('inf')

def canonical(value):
    # Sama dengan lib/digital-checkup/canonical.ts: kunci objek diurutkan, tanpa spasi.
    if isinstance(value, dict):
        return '{' + ','.join(json.dumps(k, ensure_ascii=False) + ':' + canonical(value[k]) for k in sorted(value)) + '}'
    if isinstance(value, list):
        return '[' + ','.join(canonical(v) for v in value) + ']'
    if isinstance(value, float):
        assert math.isfinite(value)
        return json.dumps(int(value)) if value.is_integer() else repr(value)
    return json.dumps(value, ensure_ascii=False)

def input_hash(value):
    return hashlib.sha256(canonical(value).encode('utf-8')).hexdigest()

def spearman(a, b):
    def mid(v):
        return [1 + sum(y < x - EPS for y in v) + (sum(abs(y - x) <= EPS for y in v) - 1) / 2 for x in v]
    x, y = mid(a), mid(b)
    mx, my = sum(x) / len(x), sum(y) / len(y)
    den = math.sqrt(sum((v - mx) ** 2 for v in x) * sum((v - my) ** 2 for v in y))
    return None if den <= EPS else sum((x[i] - mx) * (y[i] - my) for i in range(len(x))) / den

def compare_rankings(system, ranks):
    expert = [ranks[m] for m in M]
    assert sorted(expert) == list(range(1, 8)), 'Peringkat ahli harus unik 1-7.'
    top3 = [x['code'] for x in system['ranking'] if x['rank'] <= 3]
    return {'firstInTopSet': M[expert.index(1)] in system['topSet'],
            'overlapTop3': sum(1 for i, m in enumerate(M) if expert[i] <= 3 and m in top3),
            'spearman': spearman(system['rank'], expert)}

def mean(values):
    return sum(values) / len(values) if values else None

def fmt(x, digits=10):
    if x is None:
        return '-'
    if isinstance(x, bool):
        return 'ya' if x else 'tidak'
    if isinstance(x, float):
        return ('%.' + str(digits) + 'f') % x
    return str(x)

def table(headers, rows):
    cells = [[str(h) for h in headers]] + [[fmt(c) for c in row] for row in rows]
    widths = [max(len(r[i]) for r in cells) for i in range(len(headers))]
    line = lambda r: ' | '.join(c.ljust(widths[i]) for i, c in enumerate(r))
    print(line(cells[0]))
    print('-+-'.join('-' * w for w in widths))
    for r in cells[1:]:
        print(line(r))
    print()

## 2. Muat file ekspor

In [ ]:
SOURCE = os.environ.get('DL_COLAB_JSON', '')   # opsional: path file saat dijalankan di luar Colab
if SOURCE:
    with open(SOURCE, encoding='utf-8') as fh:
        bundle = json.load(fh)
else:
    from google.colab import files   # hanya tersedia di Google Colab
    uploaded = files.upload()
    bundle = json.loads(next(iter(uploaded.values())))

assert bundle.get('schemaVersion') in ('DL-COLAB-1.0', 'DL-COLAB-1.1'), 'File bukan ekspor DL-COLAB dari admin Digital Checkup.'
assert bundle['dimensions'] == D and bundle['modules'] == M, 'Urutan kode dimensi/modul berbeda dari C1-C8 / A1-A7.'
DIM_TITLES = bundle.get('dimensionTitles', {})
MODULE_NAMES = bundle.get('moduleNames', {})
records = bundle['records']
ranked = [r for r in records if r.get('run') and r['run']['status'] == 'RANKED']
print('Skema               :', bundle['schemaVersion'])
print('Record UMKM final   :', len(records))
print('Record RANKED       :', len(ranked))
print('Draf dilewati       :', len(bundle.get('skippedRecords', [])))

## 3. Tabel model: CR ahli, bobot `w`, dan matriks `R`

In [ ]:
# Bobot w, matriks R, dan CR dihitung ulang dari penilaian ahli di input canonical setiap model.
model_inputs = {m['id']: m for m in bundle.get('modelInputs', [])}
models = {}
for r in records:
    if r.get('canonicalInput'):
        models.setdefault(r['canonicalInput']['modelVersion'], (r['canonicalInput']['experts'], r['run']['modelInputId'] if r.get('run') else None))

model_checks = []
for version, (experts, model_input_id) in sorted(models.items()):
    model = build_model(experts)
    print('=' * 72)
    print('Model', version)
    print('=' * 72)
    print('Konsistensi penilaian pairwise per ahli (CR = CI / 1,41; diterima jika CR <= 0,10):')
    rows = [['Ahli %d' % (i + 1), c['lambdaMax'], c['CI'], c['CR'], c['consistent']] for i, c in enumerate(model['individualCR'])]
    rows.append(['Gabungan (GM)', model['groupCR']['lambdaMax'], model['groupCR']['CI'], model['groupCR']['CR'], model['groupCR']['consistent']])
    table(['Ahli', 'lambda max', 'CI', 'CR', 'Konsisten'], rows)
    print('Bobot umum dimensi w (Fuzzy AHP, centroid lalu normalisasi):')
    rank_w = [1 + sum(x > model['w'][j] + EPS for x in model['w']) for j in range(8)]   # minimum rank untuk bobot setara
    table(['Kode', 'Dimensi', 'l', 'm', 'u', 'w', 'Peringkat'],
          [[D[j], DIM_TITLES.get(D[j], ''), model['fuzzyWeights'][j][0], model['fuzzyWeights'][j][1], model['fuzzyWeights'][j][2], model['w'][j], rank_w[j]] for j in range(8)])
    print('Matriks relevansi modul-dimensi R (rata-rata tiga ahli, skala 0-4):')
    table(['Modul'] + D, [[m + ' ' + MODULE_NAMES.get(m, '')[:28]] + [fmt(x, 4) for x in model['R'][i]] for i, m in enumerate(M)])
    web = model_inputs.get(model_input_id, {}).get('trace')
    if web:
        diff = max(max_abs_diff(model['w'], web['w']), max_abs_diff(model['R'], web['R']),
                   max_abs_diff([c['CR'] for c in model['individualCR']], [c['CR'] for c in web['individualCR']]))
        model_checks.append((version, diff))
        print('Selisih mutlak maksimum terhadap model web (w, R, CR): %.3e -> %s\n' % (diff, 'LULUS' if diff <= TOL else 'GAGAL'))

## 4. Validasi setiap record RANKED

In [ ]:
# Validasi setiap record: hitung ulang TOPSIS dari input canonical lalu bandingkan dengan trace web.
summary_rows, failures = [], []
for r in records:
    code, run = r['umkmCode'], r.get('run')
    if not run or not r.get('canonicalInput'):
        summary_rows.append([code, 'BELUM ADA RUN', None, None, None, '-', 'tidak dinilai'])
        continue
    inp, web = r['canonicalInput'], run['trace']
    assert inp['dimensionCodes'] == D and inp['moduleCodes'] == M
    model = build_model(inp['experts'])
    mine = topsis(inp['answers'], model['w'], model['R'])
    if run['status'] != 'RANKED':
        # Tidak ada ranking untuk dibandingkan, tetapi status khususnya harus tetap sama dengan web.
        if mine['status'] != run['status']:
            failures.append(code)
        summary_rows.append([code, run['status'], None, None, None, '-', 'tidak dirangking' if mine['status'] == run['status'] else 'GAGAL (status beda)'])
        continue
    diffs = {k: max_abs_diff(mine[k], web[k]) for k in NUMERIC_KEYS}
    worst = max(diffs.values())
    same_order = [x['code'] for x in mine['ranking']] == [x['code'] for x in web['ranking']]
    exact = mine['status'] == web['status'] and mine['rank'] == web['rank'] and sorted(mine['topSet']) == sorted(web['topSet']) and same_order
    hash_ok = input_hash(inp) == run['inputHash']
    answers_ok = inp['answers'] == r['answers']
    passed = worst <= TOL and exact and hash_ok and answers_ok
    if not passed:
        failures.append(code)
    print('%s  status=%s  max|selisih|=%.3e  hash=%s  ranking=%s  -> %s' % (code, mine['status'], worst, 'cocok' if hash_ok else 'BEDA', 'sama' if exact else 'BEDA', 'LULUS' if passed else 'GAGAL'))
    if not passed:
        for k, v in diffs.items():
            if v > TOL:
                print('    %-10s selisih %.3e melebihi %.0e' % (k, v, TOL))
        if not answers_ok:
            print('    jawaban di input canonical berbeda dari jawaban record')
    summary_rows.append([code, run['status'], '%.2e' % worst, hash_ok, exact, ', '.join(mine['topSet']), 'LULUS' if passed else 'GAGAL'])

## 5. Metrik validasi tahap 2 (ranking ahli vs sistem)
Dihitung ulang dari ranking ahli yang benar-benar dikirim. Jika belum ada data, sel ini hanya memberi tahu.

In [ ]:
# Validasi tahap 2: hitung ulang metrik ranking sistem vs ranking ahli (data nyata dari ekspor).
by_assessment = {r['assessmentId']: r for r in records}
pairs, abstain, stage2_problems = [], 0, []
for v in bundle.get('validations', []):
    if v.get('status') != 'submitted':
        continue
    if v.get('decision') == 'Tidak dapat menilai':
        abstain += 1
        continue
    r = by_assessment.get(v['assessmentId'])
    if not (v.get('ranks') and r and r.get('run') and r['run']['status'] == 'RANKED' and v.get('runId') == r['run']['id']):
        continue
    inp = r['canonicalInput']
    model = build_model(inp['experts'])
    mine = compare_rankings(topsis(inp['answers'], model['w'], model['R']), v['ranks'])
    web = v.get('metrics') or {}
    same = (mine['firstInTopSet'] == web.get('firstInTopSet') and mine['overlapTop3'] == web.get('overlapTop3')
            and ((mine['spearman'] is None and web.get('spearman') is None) or
                 (mine['spearman'] is not None and web.get('spearman') is not None and abs(mine['spearman'] - web['spearman']) <= TOL)))
    if not same:
        stage2_problems.append('metrik %s x %s' % (r['umkmCode'], v['expertCode']))
    pairs.append(dict(mine, umkm=r['umkmCode'], expert=v['expertCode']))

def stats(rows):
    rho = [p['spearman'] for p in rows if p['spearman'] is not None]
    return [len(rows), mean(rho), '%d/%d' % (len(rho), len(rows)),
            (sum(p['firstInTopSet'] for p in rows) / len(rows)) if rows else None,
            mean([p['overlapTop3'] for p in rows])]

if not pairs and not abstain:
    print('Belum ada penilaian tahap 2 yang final. Kumpulkan dulu ranking ahli di /ahli (target 10 UMKM x 3 ahli).')
else:
    complete = sum(1 for u in set(p['umkm'] for p in pairs) if len(set(p['expert'] for p in pairs if p['umkm'] == u)) >= 3)
    print('Target 10 UMKM x 3 ahli: %d/10 UMKM lengkap, %d/30 pasangan sah, %d abstain.' % (complete, len(pairs), abstain))
    print('UMKM tervalidasi: %d | ahli: %d\n' % (len(set(p['umkm'] for p in pairs)), len(set(p['expert'] for p in pairs))))
    headers = ['Kelompok', 'Pasangan sah', 'Rata-rata Spearman', 'Spearman terdefinisi', 'Top-set hit rate', 'Rata-rata overlap top-3']
    rows = [['Semua ahli'] + stats(pairs)] + [[e] + stats([p for p in pairs if p['expert'] == e]) for e in sorted(set(p['expert'] for p in pairs))]
    table(headers, rows)
    web_summary = bundle.get('validationSummary')
    if web_summary:
        agg = stats(pairs)
        checks = [web_summary['validPairs'] == agg[0], web_summary['abstentions'] == abstain,
                  max_abs_diff(agg[1], web_summary['meanSpearman']) <= TOL if agg[1] is not None else web_summary['meanSpearman'] is None]
        print('Ringkasan web sama dengan hitung ulang:', 'ya' if all(checks) else 'TIDAK')
        if not all(checks):
            stage2_problems.append('ringkasan tahap 2')
    print('Metrik tahap 2 sama dengan web:', 'ya' if not stage2_problems else 'TIDAK (%s)' % ', '.join(stage2_problems))

## 6. Ringkasan akhir

In [ ]:
# Ringkasan akhir
print('Ringkasan validasi per UMKM (toleransi selisih mutlak %.0e):' % TOL)
table(['UMKM', 'Status', 'Selisih maks', 'Hash cocok', 'Ranking sama', 'Top set', 'Hasil'], summary_rows)
for item in bundle.get('skippedRecords', []):
    print('Dilewati (bukan record final):', item['assessmentId'][:8], item['status'], '%d/32 jawaban' % item['answered'])
for version, diff in model_checks:
    if diff > TOL:
        failures.append('model ' + version)
failures.extend('tahap 2 ' + x for x in globals().get('stage2_problems', []))
validated = sum(1 for row in summary_rows if row[-1] in ('LULUS', 'GAGAL'))
assert validated > 0, 'Tidak ada record RANKED untuk divalidasi. Ekspor UMKM yang hasilnya sudah RANKED.'
assert not failures, 'VALIDASI GAGAL: ' + ', '.join(failures)
print('\nVALIDASI LULUS — %d record RANKED identik dengan web sampai selisih mutlak <= %.0e.' % (validated, TOL))